# FLAN-T5: four input limits and fixed GPT silver references

This notebook uses one frozen, cleaned source package and the current approved prompt. It can generate FLAN summaries with **512, 1,024, 2,048, and 3,072 total input tokens**, with **64 maximum new output tokens** for all configurations, then join the same manually prepared GPT reference by article ID.

Each comparison CSV contains exactly:

`id,title,summary,reference_summary`

The six development/pilot articles remain candidates; they are not automatically excluded. Source-quality review and your final inclusion decisions determine the cohort. GPT summaries are **silver references**, not human gold labels. ROUGE and BERTScore measure agreement with these references, not factual correctness or beginner readability.

**Nothing expensive runs by default:** generation and scoring each require a separate explicit opt-in below. No scraping, database operations, or production pipeline changes occur here. The realtime integration of `article_preprocessing.py` remains a separate **high-priority pending task**.

## 1. Complete the source review and GPT references first

Use the frozen package under `notebook/evaluation_data/flan_reference_package_v1/` and the [workflow README](../FLAN_REFERENCE_WORKFLOW_README.md). Review `source_audit.csv`: change every `review_decision` from `pending` to `include` or `exclude`, documenting exclusions in `review_notes`. Then fill `reference_summary` in `gpt_reference_summaries.csv` for every included article. Retain all 73 rows and IDs in both master files; excluded rows may keep blank references. Quality flags are review aids, not automatic exclusions.

For every included article:

1. Use the packaged prompt **unchanged**, replacing `{article_text}` with that row's complete cleaned `article_text` (title + full body).
2. Do not show GPT any FLAN output. Use the same GPT model/version and procedure consistently and record them in the package's reference metadata.
3. Ask for the prompt's **1–2 sentences**; do not force GPT to use a numerical 64-token budget, because its tokenizer differs from FLAN's.
4. Check the answer against the source and put it in that article's `reference_summary` field. Keep the ID and source text unchanged.

One reference is reused for all four FLAN input limits. Do not write four different references. The full source remains in the master file; it does not need to be copied into the four comparison CSVs.

Install dependencies manually in the notebook kernel's environment if needed:

```bash
python -m pip install -r requirements.txt
python -m pip install -r notebook/requirements_flan_evaluation.txt
```

When using the notebook terminal from another directory, adjust those paths. No cell installs packages automatically.

In [ ]:
from datetime import datetime
from pathlib import Path
from uuid import uuid4
from zoneinfo import ZoneInfo
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Support launching Jupyter from the repo root or any nested folder."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "flan_reference_workflow.py").is_file() and (
            candidate / "ai_models" / "flan_t5.py"
        ).is_file():
            return candidate
    raise RuntimeError(
        "Project root not found. Open this notebook within the Pragma repository "
        "containing flan_reference_workflow.py."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from flan_reference_workflow import (
    DEFAULT_PACKAGE,
    generate_comparison_csvs,
    load_comparison_csvs,
    load_package,
    validate_ready,
)


def project_path(value: str | Path) -> Path:
    value = Path(value)
    return value.resolve() if value.is_absolute() else (PROJECT_ROOT / value).resolve()


PACKAGE_DIR = project_path(DEFAULT_PACKAGE)
# Optional completed copies; None means use the package defaults.
REFERENCE_CSV: str | Path | None = None
AUDIT_CSV: str | Path | None = None

# Both are intentionally False. Run All does NOT load FLAN or BERTScore models.
RUN_GENERATION = False
RUN_SCORING = False

# Completed all-73 as-is comparison; loading it is read-only. Use None for a new generation run.
EXISTING_RUN_DIR: str | Path | None = "output/flan_evaluation/input_comparison/run_20261025_012611_as_is_39b89baf"
OUTPUT_ROOT = PROJECT_ROOT / "output" / "flan_evaluation" / "input_comparison"
DEVICE_NAME = "auto"

reference_path = project_path(REFERENCE_CSV) if REFERENCE_CSV is not None else None
audit_path = project_path(AUDIT_CSV) if AUDIT_CSV is not None else None

# Reset all state when this configuration cell is rerun.
package_manifest = None
ready_rows = None
readiness_error = None
generation_result = None
run_dir = None
comparison_frames = {}
rouge_frames = {}
bertscore_frames = {}
bertscore_hashes = {}
evaluation_summary_df = None

print("Project:", PROJECT_ROOT)
print("Package:", PACKAGE_DIR)
print("Generate:", RUN_GENERATION, "| Score:", RUN_SCORING)

## 2. Validate readiness without loading a model

This read-only check requires valid source-review decisions, complete references for the selected cohort, unique IDs, and unchanged frozen source fields. A pending result is expected until you finish the manual work.

Do not bypass a failed check by dropping rows or filling placeholder references. Amend the review/reference files, then rerun this cell. Empty references, duplicate IDs, or mismatched article identities must be addressed explicitly.

In [ ]:
package_manifest = None
ready_rows = None
readiness_error = None

try:
    package_manifest = load_package(PACKAGE_DIR)
    ready_rows = validate_ready(
        PACKAGE_DIR,
        reference_csv=reference_path,
        audit_csv=audit_path,
    )
except (FileNotFoundError, ValueError, RuntimeError) as exc:
    readiness_error = str(exc)
    print("NOT READY — no generation or scoring started.")
    print(readiness_error)
    print("Finish the package's source review and GPT references, then rerun this cell.")
else:
    print(f"READY: {len(ready_rows)} included articles with valid fixed references.")
    ready_df = pd.DataFrame(ready_rows)
    display(ready_df[["id", "title", "reference_summary", "is_pilot"]].head())

## 3. Generate four FLAN comparison CSVs — explicit opt-in

Set `RUN_GENERATION = True` in the configuration cell only after completing the manual package, then rerun configuration, readiness, and this cell. This will load the existing pretrained FLAN model once, process the same selected article text for each limit, and save a new run directory.

The generation service retains the approved prompt and production model behavior, with fixed `max_new_tokens=64`, deterministic decoding, four beams, early stopping, and the existing no-repeat setting. The input ceilings count the **complete tokenized prompt**, not just the article body. This experiment does not change the production settings.

Input truncation is deliberate in this experiment. The current right-truncation behavior can remove the trailing `Summary:` marker at lower limits; it is preserved and documented rather than silently changed.

For a new run, leave `EXISTING_RUN_DIR = None`. To reuse a saved run, leave generation disabled and enter the exact existing run folder instead. References, IDs, and titles are joined and validated by ID, not by row position.

In [ ]:
generation_result = None
run_dir = None
comparison_frames = {}
rouge_frames = {}
bertscore_frames = {}
bertscore_hashes = {}

if RUN_GENERATION and EXISTING_RUN_DIR is not None:
    raise RuntimeError(
        "Choose one mode: new generation (EXISTING_RUN_DIR=None) or "
        "existing results (RUN_GENERATION=False)."
    )

if RUN_GENERATION:
    try:
        ready_rows = validate_ready(
            PACKAGE_DIR,
            reference_csv=reference_path,
            audit_csv=audit_path,
        )
    except (FileNotFoundError, ValueError, RuntimeError) as exc:
        raise RuntimeError(
            "Generation blocked. Complete source-review decisions and GPT references "
            "in the frozen package before enabling generation. Details: " + str(exc)
        ) from exc

    # Every run gets a fresh target; no previously generated CSV is overwritten.
    run_name = datetime.now(ZoneInfo("Asia/Jakarta")).strftime("run_%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
    requested_run_dir = OUTPUT_ROOT / run_name
    generation_result = generate_comparison_csvs(
        PACKAGE_DIR,
        requested_run_dir,
        reference_csv=reference_path,
        audit_csv=audit_path,
        device_name=DEVICE_NAME,
    )
    run_dir = Path(generation_result["run_dir"]).resolve()
    print("Generation complete. Run folder:", run_dir)
    for limit, csv_file in generation_result["csv_paths"].items():
        print(f"{int(limit):,} input tokens: {csv_file}")
elif EXISTING_RUN_DIR is not None:
    run_dir = project_path(EXISTING_RUN_DIR)
    print("Use saved run:", run_dir)
else:
    print("Generation skipped (RUN_GENERATION=False). No model was loaded.")
    print("Finish the manual package first, then enable generation explicitly.")

## 4. Load and validate all four saved CSVs

The loader checks the exact four-column schema, nonempty summaries, unique matching IDs, and identical titles/references against the run's frozen inputs and package. Missing or inconsistent files stop the workflow instead of being silently skipped.

An existing run can be loaded without regenerating summaries. Keep the original run manifest and frozen inputs beside its CSVs.

In [ ]:
comparison_frames = {}

if run_dir is None:
    print("No run selected. Generation and scoring remain skipped.")
else:
    loaded_comparisons = load_comparison_csvs(PACKAGE_DIR, run_dir)
    comparison_frames = {
        int(limit): pd.DataFrame(rows)
        for limit, rows in loaded_comparisons.items()
    }
    if set(comparison_frames) != {512, 1024, 2048, 3072}:
        raise RuntimeError("Expected exactly four input limits: 512, 1024, 2048, 3072.")
    print("Validated comparison pairs:")
    for limit in sorted(comparison_frames):
        print(f"  input={limit:,}: {len(comparison_frames[limit])} articles")
    display(comparison_frames[512].head())

## 5. ROUGE evaluation — separate cell

Enable `RUN_SCORING` only when you intentionally want to compute metrics. This cell calculates per-article **ROUGE-1, ROUGE-2, and ROUGE-L F1** with stemming, preserving the earlier notebook's scoring settings.

These scores measure word/sequence overlap with GPT silver references. A low score does not by itself prove the summary is incorrect, and a high score does not establish factual accuracy or accessibility.

In [ ]:
rouge_frames = {}

if not RUN_SCORING:
    print("ROUGE skipped (RUN_SCORING=False).")
else:
    if not comparison_frames:
        raise RuntimeError(
            "ROUGE blocked: load a validated comparison run in section 4 first."
        )
    from rouge_score import rouge_scorer

    rouge_scorer_instance = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True,
    )
    for limit, comparison_df in sorted(comparison_frames.items()):
        rows = []
        for article in comparison_df.to_dict("records"):
            scores = rouge_scorer_instance.score(
                target=article["reference_summary"],
                prediction=article["summary"],
            )
            rows.append({
                "id": article["id"],
                "title": article["title"],
                "rouge1_f1": scores["rouge1"].fmeasure,
                "rouge2_f1": scores["rouge2"].fmeasure,
                "rougeL_f1": scores["rougeL"].fmeasure,
            })
        rouge_frames[limit] = pd.DataFrame(rows)
        print(f"Input limit {limit:,} — {len(rows)} pairs; average ROUGE F1:")
        display(
            rouge_frames[limit][["rouge1_f1", "rouge2_f1", "rougeL_f1"]]
            .mean().rename("mean_f1").round(4)
        )

## 6. BERTScore evaluation — separate cell

This cell may download/load `roberta-large` when explicitly enabled. It preserves the previous notebook settings: English, CPU, batch size 4, `idf=False`, and `rescale_with_baseline=False`. The returned configuration hash is saved with each result set.

Per-article precision, recall, and F1 estimate semantic agreement with the same fixed GPT references. Interpret them alongside source-based factuality and readability checks, not as a standalone quality verdict.

In [ ]:
bertscore_frames = {}
bertscore_hashes = {}

if not RUN_SCORING:
    print("BERTScore skipped (RUN_SCORING=False). No scoring model was loaded.")
else:
    if not comparison_frames:
        raise RuntimeError(
            "BERTScore blocked: load a validated comparison run in section 4 first."
        )
    from bert_score import score

    for limit, comparison_df in sorted(comparison_frames.items()):
        (precision, recall, f1), configuration_hash = score(
            cands=comparison_df["summary"].tolist(),
            refs=comparison_df["reference_summary"].tolist(),
            lang="en",
            model_type="roberta-large",
            device="cpu",
            batch_size=4,
            idf=False,
            rescale_with_baseline=False,
            return_hash=True,
            verbose=True,
        )
        metric_df = comparison_df[["id", "title"]].copy()
        metric_df["bertscore_precision"] = precision.cpu().numpy()
        metric_df["bertscore_recall"] = recall.cpu().numpy()
        metric_df["bertscore_f1"] = f1.cpu().numpy()
        bertscore_frames[limit] = metric_df
        bertscore_hashes[limit] = configuration_hash
        print(f"Input limit {limit:,} — average BERTScore:")
        display(
            metric_df[["bertscore_precision", "bertscore_recall", "bertscore_f1"]]
            .mean().round(4)
        )
        print("Configuration:", configuration_hash)

## 7. Export separate metric files and the four-limit comparison

This optional export runs only after both metric cells have completed with scoring enabled. Each export uses a new timestamped folder inside the generation run's `evaluation_results/`, so rerunning scoring does not overwrite older scores.

Inside that folder:

- `input_512/`, `input_1024/`, `input_2048/`, `input_3072/`: separate ROUGE CSV, BERTScore CSV, and BERTScore configuration hash.
- `input_limit_comparison.csv`: per-limit article count and mean metrics.

Keep the unrounded values for analysis; rounding above is for display only. These metrics do not replace the planned manual factuality/readability review. The six pilot articles' prior use to choose the output budget must remain documented when reporting results.

**Completed scoring (25 October 2026):** all four limits were scored on the same 73 articles and unchanged GPT references, without regenerating FLAN. Open the [saved comparison CSV](../output/flan_evaluation/input_comparison/run_20261025_012611_as_is_39b89baf/evaluation_results/scores_20261025_023043_e19f4b6b/input_limit_comparison.csv) or [result README](../output/flan_evaluation/input_comparison/run_20261025_012611_as_is_39b89baf/evaluation_results/scores_20261025_023043_e19f4b6b/README.md) without enabling scoring again. Separate per-article ROUGE and BERTScore CSVs are in each input-limit subfolder. Saved switches remain False.

In [ ]:
evaluation_summary_df = None

if not RUN_SCORING:
    print("Metric export skipped (RUN_SCORING=False).")
else:
    expected_limits = {512, 1024, 2048, 3072}
    if (
        run_dir is None
        or set(rouge_frames) != expected_limits
        or set(bertscore_frames) != expected_limits
        or set(bertscore_hashes) != expected_limits
    ):
        raise RuntimeError(
            "Export blocked: successfully run both ROUGE and BERTScore cells "
            "for all four limits first."
        )

    evaluation_name = (
        datetime.now(ZoneInfo("Asia/Jakarta")).strftime("scores_%Y%m%d_%H%M%S") + "_" + uuid4().hex[:8]
    )
    evaluation_dir = run_dir / "evaluation_results" / evaluation_name
    evaluation_dir.mkdir(parents=True, exist_ok=False)
    mean_rows = []

    for limit in sorted(expected_limits):
        metric_dir = evaluation_dir / f"input_{limit}"
        metric_dir.mkdir(exist_ok=False)
        rouge_frames[limit].to_csv(metric_dir / "rouge_scores.csv", index=False)
        bertscore_frames[limit].to_csv(metric_dir / "bertscore_scores.csv", index=False)
        (metric_dir / "bertscore_config.txt").write_text(
            bertscore_hashes[limit] + "\n", encoding="utf-8"
        )
        mean_rows.append({
            "max_input_tokens": limit,
            "article_count": len(comparison_frames[limit]),
            **rouge_frames[limit][["rouge1_f1", "rouge2_f1", "rougeL_f1"]]
            .mean().to_dict(),
            **bertscore_frames[limit][
                ["bertscore_precision", "bertscore_recall", "bertscore_f1"]
            ].mean().to_dict(),
        })

    evaluation_summary_df = pd.DataFrame(mean_rows)
    evaluation_summary_df.to_csv(
        evaluation_dir / "input_limit_comparison.csv", index=False
    )
    display(evaluation_summary_df.round(4))
    print("Metric results saved to:", evaluation_dir)